In [5]:
!pip install pandas matplotlib seaborn tenacity

In [6]:
import pandas as pd
import json
import os
import urllib.request

# Configure the local Ollama client.
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2")

In [7]:
tasks = [
    {
        "task_id": 1,
        "category": "Factual",
        "prompt": "What is the exact height of Mount Everest in meters according to the official 2020 joint announcement by Nepal and China?",
        "ground_truth": "8848.86 meters"
    },
    {
        "task_id": 2,
        "category": "Reasoning",
        "prompt": "If a clock strikes once at 1 o'clock, twice at 2 o'clock, and so on, how many times will it strike in total in a single 12-hour cycle?",
        "ground_truth": "78 times (1+2+3+4+5+6+7+8+9+10+11+12 = 78)"
    },
    {
        "task_id": 3,
        "category": "Comprehension",
        "prompt": "Summarize the core argument in one sentence: 'Although social media platforms facilitate unprecedented global communication, they simultaneously foster echo chambers that polarize public opinion and degrade democratic discourse.'",
        "ground_truth": "While social media connects people globally, it also creates echo chambers that polarize public opinion and harm democratic discourse."
    },
    {
        "task_id": 4,
        "category": "Factual / QA",
        "prompt": "Who wrote the novel 'One Hundred Years of Solitude' and in what year was it first published?",
        "ground_truth": "Gabriel García Márquez, first published in 1967."
    },
    {
        "task_id": 5,
        "category": "Reasoning",
        "prompt": "A box contains 3 red apples and 7 green apples. If you pick two apples at random without replacement, what is the probability that both are red?",
        "ground_truth": "1/15 (or approx 6.67%). Calculation: (3/10) * (2/9) = 6/90 = 1/15."
    }
]

df_tasks = pd.DataFrame(tasks)
display(df_tasks)

,task_id,category,prompt,ground_truth
0,1,Factual,What is the exact height of Mount Everest in meters according to the official 2020 joint announcement by Nepal and China?,8848.86 meters
1,2,Reasoning,"If a clock strikes once at 1 o'clock, twice at 2 o'clock, and so on, how many times will it strike in total in a single 12-hour cycle?",78 times (1+2+3+4+5+6+7+8+9+10+11+12 = 78)
2,3,Comprehension,"Summarize the core argument in one sentence: 'Although social media platforms facilitate unprecedented global communication, they simultaneously foster echo chambers that polarize public opinion and degrade democratic discourse.'","While social media connects people globally, it also creates echo chambers that polarize public opinion and harm democratic discourse."
3,4,Factual / QA,Who wrote the novel 'One Hundred Years of Solitude' and in what year was it first published?,"Gabriel García Márquez, first published in 1967."
4,5,Reasoning,"A box contains 3 red apples and 7 green apples. If you pick two apples at random without replacement, what is the probability that both are red?",1/15 (or approx 6.67%). Calculation: (3/10) * (2/9) = 6/90 = 1/15.


In [8]:
import time
import pandas as pd
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception

responses = []
target_model = OLLAMA_MODEL

# Helper function to retry on 503 (Unavailable) and 429 (Rate Limits)
@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=2, min=1, max=5),
    reraise=True
)
def generate_with_retry(prompt):
    request_body = json.dumps({
        "model": target_model,
        "prompt": prompt,
        "stream": False,
    }).encode("utf-8")
    request = urllib.request.Request(
        f"{OLLAMA_BASE_URL}/api/generate",
        data=request_body,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=120) as response:
        payload = json.loads(response.read().decode("utf-8"))
    answer = payload.get("response", "").strip()
    if not answer:
        raise RuntimeError("Ollama returned an empty response")
    return answer

print("Querying the target model...")
for i, task in enumerate(tasks):
    # minimal delay to stay safe on free limits
    if i > 0:
        time.sleep(1)
    
    try:
        model_text = generate_with_retry(task["prompt"])
        status = "Success"
        print(f"Successfully retrieved response for task {task['task_id']}")
    except Exception as e:
        model_text = f"Error: {str(e)}"
        status = "Failed"
        print(f"Failed to retrieve response for task {task['task_id']}: {str(e)}")
    
    responses.append({
        "Task ID": task["task_id"],
        "Category": task["category"],
        "Prompt": task["prompt"],
        "Ground Truth": task["ground_truth"],
        "Model Response": model_text,
        "Status": status
    })

# Create DataFrame
df_eval = pd.DataFrame(responses)

# Calculate and print the error rate
total_requests = len(df_eval)
failed_requests = len(df_eval[df_eval["Status"] == "Failed"])
error_rate = (failed_requests / total_requests) * 100

print("\n=== Execution Summary ===")
print(f"Total Requests: {total_requests}")
print(f"Failed Requests: {failed_requests}")
print(f"Error Rate: {error_rate:.2f}%")
print("=========================\n")

pd.set_option('display.max_colwidth', None)
display(df_eval)

Querying the target model...
Successfully retrieved response for task 1
Successfully retrieved response for task 2
Successfully retrieved response for task 3
Successfully retrieved response for task 4
Successfully retrieved response for task 5

=== Execution Summary ===
Total Requests: 5
Failed Requests: 0
Error Rate: 0.00%



,Task ID,Category,Prompt,Ground Truth,Model Response,Status
0,1,Factual,What is the exact height of Mount Everest in meters according to the official 2020 joint announcement by Nepal and China?,8848.86 meters,"The official height of Mount Everest is 8,848.86 meters above sea level, according to the 2020 joint announcement by Nepal and China.",Success
1,2,Reasoning,"If a clock strikes once at 1 o'clock, twice at 2 o'clock, and so on, how many times will it strike in total in a single 12-hour cycle?",78 times (1+2+3+4+5+6+7+8+9+10+11+12 = 78),"To calculate the total number of times the clock strikes, we need to find the sum of an arithmetic series.\n\nLet's list the number of times the clock strikes in a 12-hour cycle:\n\n1. At 1 o'clock, it strikes 1 time\n2. At 2 o'clock, it strikes 2 times (1 + 1)\n3. At 3 o'clock, it strikes 3 times (1 + 2)\n4. At 4 o'clock, it strikes 4 times (1 + 2 + 3)\n5. At 5 o'clock, it strikes 5 times (1 + 2 + 3 + 4)\n6. At 6 o'clock, it strikes 6 times (1 + 2 + 3 + 4 + 5)\n7. At 7 o'clock, it strikes 7 times (1 + 2 + 3 + 4 + 5 + 6)\n8. At 8 o'clock, it strikes 8 times (1 + 2 + 3 + 4 + 5 + 6 + 7)\n9. At 9 o'clock, it strikes 9 times (1 + 2 + 3 + 4 + 5 + 6 + 7 + 8)\n10. At 10 o'clock, it strikes 10 times (1 + 2 + 3 + 4 + 5 + 6 + 7 + 8 + 9)\n11. At 11 o'clock, it strikes 11 times (1 + 2 + 3 + 4 + 5 + 6 + 7 + 8 + 9 + 10)\n12. At 12 o'clock, it strikes 12 times (1 + 2 + 3 + 4 + 5 + 6 + 7 + 8 + 9 + 10 + 11)\n\nThe sum of the first 12 natural numbers is:\n\n1 + 2 + 3 + 4 + 5 + 6 + 7 + 8 + 9 + 10 + 11 + 12 = 78\n\nTherefore, the clock will strike a total of 78 times in a single 12-hour cycle.",Success
2,3,Comprehension,"Summarize the core argument in one sentence: 'Although social media platforms facilitate unprecedented global communication, they simultaneously foster echo chambers that polarize public opinion and degrade democratic discourse.'","While social media connects people globally, it also creates echo chambers that polarize public opinion and harm democratic discourse.","Social media platforms create both opportunities for global communication and echo chambers that reinforce existing biases, ultimately contributing to the erosion of democratic discourse and polarized public opinion.",Success
3,4,Factual / QA,Who wrote the novel 'One Hundred Years of Solitude' and in what year was it first published?,"Gabriel García Márquez, first published in 1967.",The novel 'One Hundred Years of Solitude' was written by Colombian author Gabriel García Márquez and was first published in 1967.,Success
4,5,Reasoning,"A box contains 3 red apples and 7 green apples. If you pick two apples at random without replacement, what is the probability that both are red?",1/15 (or approx 6.67%). Calculation: (3/10) * (2/9) = 6/90 = 1/15.,"To find the probability that both apples are red, we need to calculate the probability of picking a red apple first, and then picking another red apple.\n\nThere are 3 red apples and 10 apples in total (3 red + 7 green). \n\nThe probability of picking a red apple first is 3/10.\n\nIf we pick a red apple first, there are now 2 red apples and 9 apples in total (2 red + 7 green).\n\nThe probability of picking another red apple is 2/9.\n\nTo find the overall probability of picking two red apples, we multiply the probabilities of each event:\n\n(3/10) x (2/9) = 6/90 = 1/15",Success
